In [21]:
"""

This class was heavily genreates with AI.

Do not expect you to understand.

"""

from config import EXTERNAL_DATASET_INPUTS, INTERNAL_DATASET_INPUTS, DICTIONARY_DATASET_INPUTS,Columns
from data_processing.processing import TextProcessor
from transformers import AutoTokenizer
from utils import stack_csvs_from_folder
import sentencepiece as spm
import pandas as pd
import json

from transformers import MBartForConditionalGeneration , AutoTokenizer,AutoModelForSeq2SeqLM
import torch


In [22]:

def load_akkadian_text() -> pd.Series:
    """Load and concatenate internal and external datasets, return transliteration column."""
    df_internal = stack_csvs_from_folder(INTERNAL_DATASET_INPUTS)
    df_external = stack_csvs_from_folder(EXTERNAL_DATASET_INPUTS)
    df_dictionary = stack_csvs_from_folder(DICTIONARY_DATASET_INPUTS)
    df = pd.concat([df_internal, df_external,df_dictionary], ignore_index=True)

    tp = TextProcessor()

    df[Columns.TRANSLITERATION] = df[Columns.TRANSLITERATION].apply(
        lambda x: tp.preprocess_transliteration_text(
            transliteration_text=x,
            separate_compounds=False,
            with_hyphens=False,
            named_determinatives=False,
            normalize_chars=False,
            diacritic_mode=True,
        )
    )
    return df[Columns.TRANSLITERATION].dropna()

In [23]:

def inspect_corpus(akkadian_text: pd.Series):
    """Print statistics about the corpus to guide vocab size selection."""
    lines = akkadian_text.tolist()

    total_words = sum(len(line.split()) for line in lines)
    unique_words = len(set(word for line in lines for word in line.split()))

    print(f"Total entries:      {len(lines)}")
    print(f"Total words:        {total_words}")
    print(f"Unique words:       {unique_words}")
    print(f"Avg sentence length:{total_words / len(lines):.1f}")
    print(f"Suggested vocab size (total_words / 200): {int(total_words / 200)}")



In [24]:
def save_corpus_to_file(
    akkadian_text: pd.Series, output_path: str = "akkadian_corpus.txt"
) -> str:
    """SentencePiece requires a plain text file as input — one sentence per line."""
    with open(output_path, "w", encoding="utf-8") as f:
        for line in akkadian_text.tolist():
            f.write(line.strip() + "\n")
    print(f"Corpus saved to {output_path}")
    return output_path

In [25]:

def train_tokenizer(corpus_path: str, vocab_size: int = 2500):
    """Train a SentencePiece BPE model on the Akkadian corpus."""
    spm.SentencePieceTrainer.train(
        input=corpus_path,
        model_prefix="akkadian_vocab_sp",
        vocab_size=vocab_size,
        model_type="bpe",
        user_defined_symbols=["<lgg_s>", "<lgg_e>","<gap>"],
        character_coverage=1.0,  # cover ALL Akkadian characters
        split_by_whitespace=True,
        split_by_unicode_script=True,
        byte_fallback=True,  # no <unk>, rare chars -> UTF-8 bytes
        # min_sentencepiece_length removed -- not a valid TrainerSpec field
        max_sentencepiece_length=16,
        input_sentence_size=50000,
        shuffle_input_sentence=True,
        train_extremely_large_corpus=False,
        normalization_rule_name="identity",
    )
    print("Tokenizer training complete.")


In [26]:
def visualize_tokenizer(test_text: str = "<lgg_s> a-na be-el-ia <lgg_e>"):
    """Load trained model, tokenize a test string, and save full vocab to JSON."""
    sp = spm.SentencePieceProcessor(model_file="akkadian_sp.model")

    tokens = sp.encode_as_pieces(test_text)
    ids = sp.encode_as_ids(test_text)

    print(f"\nTest string: {test_text}")
    print(f"Pieces:      {tokens}")
    print(f"IDs:         {ids}")

    vocab = {sp.id_to_piece(i): sp.get_score(i) for i in range(sp.get_piece_size())}
    with open("akkadian_vocab.json", "w", encoding="utf-8") as f:
        json.dump(vocab, f, ensure_ascii=False, indent=4)

    print(f"\nFull vocab ({len(vocab)} tokens) saved to akkadian_vocab.json")
    return sp


In [27]:

def _compute_fertility(lines: list, tokenize_fn, unk_id: int | None = None) -> dict:
    """
    Generic helper -- computes fertility and UNK rate for any tokenizer.
    tokenize_fn(line) must return a list of integer token IDs.
    """
    total_words = 0
    total_tokens = 0
    unk_count = 0

    for line in lines:
        total_words += len(line.split())
        token_ids = tokenize_fn(line)
        total_tokens += len(token_ids)
        if unk_id is not None:
            unk_count += token_ids.count(unk_id)

    fertility = total_tokens / total_words if total_words > 0 else 0
    unk_rate = unk_count / total_tokens if total_tokens > 0 else 0

    return {
        "total_words": total_words,
        "total_tokens": total_tokens,
        "fertility": fertility,
        "unk_count": unk_count,
        "unk_rate": unk_rate,
    }


In [28]:
def analyze_mbart_coverage(akkadian_text: pd.Series) -> dict:
    """
    Check how well the existing mBART tokenizer handles Akkadian text.
    Returns the stats dict so compare_fertility() can reuse it without re-running.
    """
    tokenizer = AutoTokenizer.from_pretrained(
        "facebook/mbart-large-50-many-to-many-mmt"
    )
    lines = akkadian_text.tolist()

    stats = _compute_fertility(
        lines,
        tokenize_fn=lambda line: tokenizer(line, add_special_tokens=False)["input_ids"],
        unk_id=tokenizer.unk_token_id,
    )

    print("\n--- mBART Coverage Analysis ---")
    print(f"Total words:      {stats['total_words']}")
    print(f"Total tokens:     {stats['total_tokens']}")
    print(
        f"Fertility:        {stats['fertility']:.2f}  (tokens per word -- lower is better)"
    )
    print(f"UNK count:        {stats['unk_count']}")
    print(f"UNK rate:         {stats['unk_rate']:.2%}")

    print("\nInterpretation:")
    if stats["fertility"] < 2.0:
        print("  [OK] Low fertility -- mBART handles your text reasonably well.")
    elif stats["fertility"] < 4.0:
        print(
            "  [!!] Medium fertility -- mBART is splitting heavily, new tokens would help."
        )
    else:
        print("  [XX] High fertility -- mBART is not suited for this language as-is.")

    if stats["unk_rate"] > 0.05:
        print(
            "  [XX] High UNK rate -- many characters are completely unknown to mBART."
        )
    else:
        print("  [OK] Low UNK rate -- mBART can at least represent most characters.")

    print("\n--- Sample Tokenizations ---")
    for line in lines[:5]:
        tokens = tokenizer.tokenize(line)
        print(f"  Input:  {line}")
        print(f"  Tokens: {tokens}\n")

    return stats


In [29]:
def compare_fertility(akkadian_text: pd.Series, n_samples: int = 5):
    """
    Side-by-side fertility comparison between the original mBART tokenizer
    and the newly trained Akkadian SentencePiece tokenizer.

    Prints aggregate stats and per-sentence examples so you can see
    exactly how much improvement the new tokenizer provides.
    """
    lines = akkadian_text.tolist()

    print("Loading tokenizers...")
    mbart_tokenizer = AutoTokenizer.from_pretrained(
        "facebook/mbart-large-50-many-to-many-mmt"
    )
    akkadian_sp = spm.SentencePieceProcessor(model_file="akkadian_sp.model")

    mbart_stats = _compute_fertility(
        lines,
        tokenize_fn=lambda line: mbart_tokenizer(line, add_special_tokens=False)[
            "input_ids"
        ],
        unk_id=mbart_tokenizer.unk_token_id,
    )

    akkadian_stats = _compute_fertility(
        lines,
        tokenize_fn=lambda line: akkadian_sp.encode_as_ids(line),
        unk_id=None,  # byte_fallback=True means no UNK in our tokenizer
    )

    # Summary table
    print("\n" + "=" * 55)
    print(f"{'Metric':<25} {'mBART':>12} {'Akkadian SP':>12}")
    print("=" * 55)
    print(
        f"{'Total tokens':<25} {mbart_stats['total_tokens']:>12,} {akkadian_stats['total_tokens']:>12,}"
    )
    print(
        f"{'Fertility (tok/word)':<25} {mbart_stats['fertility']:>12.2f} {akkadian_stats['fertility']:>12.2f}"
    )
    print(
        f"{'UNK count':<25} {mbart_stats['unk_count']:>12,} {'N/A (byte fallback)':>12}"
    )
    print(f"{'UNK rate':<25} {mbart_stats['unk_rate']:>11.2%} {'N/A':>12}")
    print("=" * 55)

    improvement = (1 - akkadian_stats["fertility"] / mbart_stats["fertility"]) * 100
    print(f"\nFertility reduction: {improvement:.1f}%")
    if improvement > 50:
        print("  [OK] Excellent -- new tokenizer dramatically reduces token count.")
    elif improvement > 20:
        print("  [!!] Moderate improvement -- consider increasing vocab_size.")
    else:
        print("  [XX] Little improvement -- may need more data or a larger vocab.")

    # Per-sentence examples
    print(f"\n--- Side-by-Side Examples (first {n_samples} sentences) ---")
    for line in lines[:n_samples]:
        mbart_tokens = mbart_tokenizer.tokenize(line)
        akkadian_tokens = akkadian_sp.encode_as_pieces(line)
        print(f"  Input:       {line}")
        print(f"  mBART:       {mbart_tokens}  ({len(mbart_tokens)} tokens)")
        print(f"  Akkadian SP: {akkadian_tokens}  ({len(akkadian_tokens)} tokens)")
        print()

    return akkadian_stats["fertility"]


In [30]:

def extend_mbart_with_akkadian(
    akkadian_model_path: str = "akkadian_sp.model",
    output_dir: str = "mbart-akkadian",
):
    """
    Extract new tokens from the Akkadian SP model, add them to mBART's tokenizer,
    resize the model's embedding matrix, and save everything to output_dir.
    """

    print("\nLoading mBART tokenizer and model...")
    tokenizer = AutoTokenizer.from_pretrained("MarkSpaghetti/mbart-lora-r32-alpha-16")
    model = AutoModelForSeq2SeqLM.from_pretrained("MarkSpaghetti/mbart-lora-r32-alpha-16")

    # Extract all tokens from the Akkadian SP model
    akkadian_sp = spm.SentencePieceProcessor(model_file=akkadian_model_path)


    AKKADIAN_SPECIAL_TOKENS = {"<lgg_s>", "<lgg_e>", "<gap>"}
    SKIP_TOKENS = {"<unk>", "<s>", "</s>"}


    special_tokens = []
    regular_tokens = []
    for i in range(akkadian_sp.get_piece_size()):
        piece = akkadian_sp.id_to_piece(i)
        if piece in SKIP_TOKENS:
            continue
        elif piece in AKKADIAN_SPECIAL_TOKENS:
            special_tokens.append(piece)
            print(piece)
        else:
            regular_tokens.append(piece) 

    akkadian_vocab = set(tokenizer.get_vocab().keys())
    new_special = [t for t in special_tokens if t not in akkadian_vocab]
    new_regular = [t for t in regular_tokens if t not in akkadian_vocab]


    print(f"Akkadian SP vocab size:  {len(akkadian_vocab)}")
    print(f"Tokens already in mBART: {len(special_tokens) + len(regular_tokens) - (len(new_special) - len(new_regular))}")
    print(f"New tokens to add:       {len(new_special) + len(new_regular)}")
    
    # Now <lgg_s>, <lgg_e>, <gap> ONLY go here, never to add_tokens
    tokenizer.add_special_tokens({"additional_special_tokens": new_special})
    # And regular BPE tokens ONLY go here, never mixed with special
    tokenizer.add_tokens(new_regular)

    # Resize model embeddings to match the new vocab size
    # New token embeddings are initialized as the mean of all existing embeddings
    # (better than random initialization for faster convergence)
    old_vocab_size = model.model.shared.weight.shape[0]
    model.resize_token_embeddings(len(tokenizer))

    with torch.no_grad():
        mean_embedding = model.model.shared.weight[:old_vocab_size].mean(dim=0)
        model.model.shared.weight[old_vocab_size:] = mean_embedding

    print(f"Embedding matrix resized: {old_vocab_size} -> {len(tokenizer)}")

    # Save both tokenizer and model

    return tokenizer, model


In [31]:


# Step 1 -- Load data
akkadian_text = load_akkadian_text()

# Step 2 -- Inspect corpus to guide vocab size choice
inspect_corpus(akkadian_text)

# Step 3 -- Analyze what mBART already covers
analyze_mbart_coverage(akkadian_text)

# Step 4 -- Save corpus to file and train tokenizer
corpus_path = save_corpus_to_file(akkadian_text)

train_tokenizer(corpus_path, vocab_size=1500)

# Step 5 -- Visualize new tokenizer
visualize_tokenizer()

# Step 6 -- Compare mBART vs new tokenizer fertility side by side
compare_fertility(akkadian_text, n_samples=3)


Total entries:      58198
Total words:        234773
Unique words:       50114
Avg sentence length:4.0
Suggested vocab size (total_words / 200): 1173

--- mBART Coverage Analysis ---
Total words:      234773
Total tokens:     1714358
Fertility:        7.30  (tokens per word -- lower is better)
UNK count:        4761
UNK rate:         0.28%

Interpretation:
  [XX] High fertility -- mBART is not suited for this language as-is.
  [OK] Low UNK rate -- mBART can at least represent most characters.

--- Sample Tokenizations ---
  Input:  <gap> <lgg_s>gín<lgg_e> <lgg_s>kù.babbar<lgg_e> ṣarupáam iṣéer šubelim šu-{d}<lgg_s>en.líl<lgg_e> išu iští <lgg_s>kù.babbar<lgg_e> uktaal qádum ṭuppìšu ša kànéeš
  Tokens: ['▁<', 'ga', 'p', '>', '▁<', 'l', 'gg', '_', 's', '>', 'g', 'ín', '<', 'l', 'gg', '_', 'e', '>', '▁<', 'l', 'gg', '_', 's', '>', 'k', 'ù', '.', 'bab', 'bar', '<', 'l', 'gg', '_', 'e', '>', '▁', 'ṣ', 'aru', 'pá', 'am', '▁i', 'ṣ', 'é', 'er', '▁', 'šu', 'beli', 'm', '▁', 'šu', '-', '{', 'd', 

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: akkadian_corpus.txt
  input_format: 
  model_prefix: akkadian_vocab_sp
  model_type: BPE
  vocab_size: 1500
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 50000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <lgg_s>
  user_defined_symbols: <lgg_e>
  user_defined_symbols: <gap>
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_pi


Metric                           mBART  Akkadian SP
Total tokens                 1,714,358      831,916
Fertility (tok/word)              7.30         3.54
UNK count                        4,761 N/A (byte fallback)
UNK rate                        0.28%          N/A

Fertility reduction: 51.5%
  [OK] Excellent -- new tokenizer dramatically reduces token count.

--- Side-by-Side Examples (first 3 sentences) ---
  Input:       <gap> <lgg_s>gín<lgg_e> <lgg_s>kù.babbar<lgg_e> ṣarupáam iṣéer šubelim šu-{d}<lgg_s>en.líl<lgg_e> išu iští <lgg_s>kù.babbar<lgg_e> uktaal qádum ṭuppìšu ša kànéeš
  mBART:       ['▁<', 'ga', 'p', '>', '▁<', 'l', 'gg', '_', 's', '>', 'g', 'ín', '<', 'l', 'gg', '_', 'e', '>', '▁<', 'l', 'gg', '_', 's', '>', 'k', 'ù', '.', 'bab', 'bar', '<', 'l', 'gg', '_', 'e', '>', '▁', 'ṣ', 'aru', 'pá', 'am', '▁i', 'ṣ', 'é', 'er', '▁', 'šu', 'beli', 'm', '▁', 'šu', '-', '{', 'd', '}', '<', 'l', 'gg', '_', 's', '>', 'en', '.', 'lí', 'l', '<', 'l', 'gg', '_', 'e', '>', '▁iš', 'u', '▁i

3.5434909465739244

In [13]:
tokenizer, model = extend_mbart_with_akkadian()


Loading mBART tokenizer and model...


Loading weights:   0%|          | 0/518 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


<lgg_s>
<lgg_e>
<gap>
Akkadian SP vocab size:  250055
Tokens already in mBART: 1789
New tokens to add:       598


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding matrix resized: 250055 -> 250653


In [8]:
model.push_to_hub("MarkSpaghetti/mbart-tokens-lora-r32-alpha-16")
tokenizer.push_to_hub("MarkSpaghetti/mbart-tokens-lora-r32-alpha-16")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/MarkSpaghetti/mbart-tokens-lora-r32-alpha-16/commit/a43c7e15467462c8528825798f1df50ca3384e9e', commit_message='Upload tokenizer', commit_description='', oid='a43c7e15467462c8528825798f1df50ca3384e9e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MarkSpaghetti/mbart-tokens-lora-r32-alpha-16', endpoint='https://huggingface.co', repo_type='model', repo_id='MarkSpaghetti/mbart-tokens-lora-r32-alpha-16'), pr_revision=None, pr_num=None)

In [37]:
from transformers import PreTrainedTokenizer
from typing import List, Optional, Dict
import re


class AkkadianMBartTokenizer(PreTrainedTokenizer):
    def __init__(self, hf_tokenizer, akkadian_sp_path, **kwargs):
        self.hf_tokenizer = hf_tokenizer
        self.akkadian_sp = spm.SentencePieceProcessor(model_file=akkadian_sp_path)
        self.SPECIAL_TOKEN_PATTERN = re.compile(r'(<lgg_s>|<lgg_e>|<gap>)')
        self.SPECIAL_TOKENS = {"<lgg_s>", "<lgg_e>", "<gap>"}

        # Build ID mappings
        mbart_vocab = hf_tokenizer.get_vocab()
        self.akk_to_mbart = {}
        for akk_id in range(self.akkadian_sp.get_piece_size()):
            piece = self.akkadian_sp.id_to_piece(akk_id)
            self.akk_to_mbart[akk_id] = mbart_vocab.get(piece, hf_tokenizer.unk_token_id)
        self.mbart_to_akk = {v: k for k, v in self.akk_to_mbart.items()}

        super().__init__(
            bos_token=hf_tokenizer.bos_token,
            eos_token=hf_tokenizer.eos_token,
            unk_token=hf_tokenizer.unk_token,
            pad_token=hf_tokenizer.pad_token,
            src_lang="ak_XX",
            **kwargs
        )

    # ---- Required overrides ----

    @property
    def vocab_size(self):
        return len(self.hf_tokenizer)

    def get_vocab(self) -> Dict[str, int]:
        return self.hf_tokenizer.get_vocab()

    def _tokenize(self, text: str) -> List[str]:
        """Text -> list of string pieces."""
        pieces = []
        for part in self.SPECIAL_TOKEN_PATTERN.split(text):
            if part in self.SPECIAL_TOKENS:
                pieces.append(part)
            elif part.strip():
                for akk_id in self.akkadian_sp.encode(part.strip()):
                    pieces.append(self.akkadian_sp.id_to_piece(akk_id))
        return pieces

    def _convert_token_to_id(self, token: str) -> int:
        return self.hf_tokenizer.convert_tokens_to_ids(token)

    def _convert_id_to_token(self, index: int) -> str:
        if index in self.mbart_to_akk:
            return self.akkadian_sp.id_to_piece(self.mbart_to_akk[index])
        return self.hf_tokenizer.convert_ids_to_tokens(index)

    def convert_tokens_to_string(self, tokens: List[str]) -> str:
        return self.akkadian_sp.decode([
            self.akkadian_sp.piece_to_id(t) for t in tokens
            if self.akkadian_sp.piece_to_id(t) != 0
        ])

    def save_vocabulary(self, save_directory: str, filename_prefix=None):
        # Delegate to HF tokenizer
        return self.hf_tokenizer.save_vocabulary(save_directory, filename_prefix)

In [38]:
new_tokenizer = AkkadianMBartTokenizer(tokenizer, "akkadian_sp.model")

text = "<lgg_s>igi<lgg_e> szalimaszur <lgg_s>dumu<lgg_e> ili2a <lgg_s>igi<lgg_e> sa2si2a"
ids = new_tokenizer(text)
print(ids)
print(new_tokenizer.convert_ids_to_tokens(ids))

{'input_ids': [250055, 17, 735, 250056, 250377, 250324, 250055, 6, 110702, 250056, 17, 150, 304, 11, 250055, 17, 735, 250056, 57, 304, 172, 304, 11], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


ValueError: invalid literal for int() with base 10: 'input_ids'

In [ ]:
tokenizer.push_to_hub("MarkSpaghetti/mbart-tokens-lora-r32-alpha-16")

In [43]:
text = "<lgg_s>igi<lgg_e> szalimaszur <lgg_s>dumu<lgg_e> ili2a"
ids = new_tokenizer(text)["input_ids"]
decoded = new_tokenizer.decode(ids, skip_special_tokens=False)
print(f"Original : {text}")
print(f"Decoded  : {decoded}")

Original : <lgg_s>igi<lgg_e> szalimaszur <lgg_s>dumu<lgg_e> ili2a
Decoded  : <lgg_s> igi<lgg_e> szalimaszur<lgg_s> dumu<lgg_e> ili2a


In [44]:
# Compare how mBART alone tokenizes vs your tokenizer
mbart_tokens = tokenizer.tokenize("szalimaszur")
akk_tokens   = new_tokenizer._tokenize("szalimaszur")
print(f"mBART alone : {mbart_tokens}")  # expect ugly fragments
print(f"Your tokenizer: {akk_tokens}")  # expect clean Akkadian subwords

mBART alone : ['szal', '▁i', 'masz', '▁ur']
Your tokenizer: ['▁szalima', 'szur']


In [16]:
tp = TextProcessor()

text = tp.preprocess_transliteration_text(
            transliteration_text="a3",
            separate_compounds=False,
            with_hyphens=False,
            named_determinatives=False,
            normalize_chars=False,
            diacritic_mode=True,
        )
    

print(text)

à
